# FFSSFI–WEUSO: Remote-Sensing Semantic Segmentation

Complete, configurable PyTorch research implementation for the five-stage workflow:
1. Remote-sensing image and mask loading
2. Radiometric preprocessing, denoising, normalization, GLCM texture and morphological profiles
3. Multi-scale CNN encoder–decoder with a selective state-space-inspired block
4. Multi-order fractional Fourier spectral–spatial feature interaction (FFSSFI)
5. Wasserstein + evidential uncertainty-guided segmentation optimization (WEUSO)

**Important research notes**
- Set the dataset paths, class names, band selection and patch size in the configuration cell before running.
- The fractional transform below is a differentiable FFT-domain fractional spectral operator. It is an efficient practical FrFT-style implementation; if your paper requires a particular canonical discrete FrFT definition, replace this operator with that exact definition and document it.
- The state-space block is an efficient selective, gated state-space-inspired implementation using depthwise sequence mixing. It is not a drop-in reproduction of a specific external SSM library.
- SCSM, PSseg-FD, CVMH-UNet and PENet are paper-specific methods. Their exact architectures cannot be inferred from acronyms alone. The notebook includes a U-Net reference baseline and a common training/evaluation interface; provide the four papers or their architecture details to implement faithful reproductions rather than inventing them.

## 1. Environment

In [ ]:
# Run once in a fresh Colab/local environment if packages are missing.
%pip install -q torch torchvision rasterio opencv-python-headless scikit-image scipy scikit-learn matplotlib pandas tqdm

## 2. Imports and reproducibility

In [ ]:
import os, glob, math, random, warnings, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support, jaccard_score
from scipy import ndimage as ndi
from skimage.feature import graycomatrix, graycoprops
from skimage.morphology import disk, opening, closing, white_tophat, black_tophat

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 3. Dataset configuration

In [ ]:
# Expected paired layout (recommended):
# DATA_ROOT/
#   images/  scene_001.tif, scene_002.tif, ...
#   masks/   scene_001.png, scene_002.png, ...
#
# Image and mask files are paired by filename stem. TIFF, PNG, JPG, JPEG and NPY are supported.
# For NPY images, expected shape is HxW or HxWxC (or CxHxW; set NPY_CHANNELS_FIRST accordingly).

DATA_ROOT = "/content/remote_sensing_dataset"  # CHANGE THIS
IMAGE_DIR = os.path.join(DATA_ROOT, "images")
MASK_DIR  = os.path.join(DATA_ROOT, "masks")

CLASS_NAMES = ["background", "building", "road", "vegetation", "water", "agriculture", "bare_land"]
NUM_CLASSES = len(CLASS_NAMES)
IGNORE_INDEX = 255
PATCH_SIZE = 256
BATCH_SIZE = 4
NUM_WORKERS = 2
EPOCHS = 100
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
NPY_CHANNELS_FIRST = False

# Set to None to use every image band. Otherwise provide zero-based band indices, e.g. [0,1,2,3].
BAND_INDICES = None
# If masks are RGB color masks, define exact RGB-to-class mapping here.
# Example: {(0,0,0):0, (255,0,0):1, (0,255,0):2}
RGB_MASK_MAP = None

# Optional preprocessing controls
APPLY_DENOISING = True
GLCM_LEVELS = 16
MORPH_RADII = [2, 4, 8]

## 4. File pairing and dataset inspection

In [ ]:
VALID_EXTS = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".npy"}

def index_files(folder):
    out = {}
    for p in Path(folder).rglob("*"):
        if p.is_file() and p.suffix.lower() in VALID_EXTS:
            out[p.stem] = str(p)
    return out

image_index, mask_index = index_files(IMAGE_DIR), index_files(MASK_DIR)
pairs = [(image_index[k], mask_index[k]) for k in sorted(image_index.keys() & mask_index.keys())]
print(f"Images: {len(image_index)} | Masks: {len(mask_index)} | Matched pairs: {len(pairs)}")
print("Unmatched images:", len(image_index.keys() - mask_index.keys()))
print("Unmatched masks:", len(mask_index.keys() - image_index.keys()))
assert len(pairs) > 0, "No matched image-mask pairs. Check IMAGE_DIR, MASK_DIR and filename stems."

## 5. Image reading and preprocessing

In [ ]:
def read_image(path):
    suffix = Path(path).suffix.lower()
    if suffix == ".npy":
        arr = np.load(path)
        if arr.ndim == 2: arr = arr[..., None]
        elif NPY_CHANNELS_FIRST and arr.ndim == 3: arr = np.moveaxis(arr, 0, -1)
    elif suffix in {".tif", ".tiff"}:
        import rasterio
        with rasterio.open(path) as src:
            arr = np.moveaxis(src.read(), 0, -1)
    else:
        import cv2
        arr = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if arr is None: raise ValueError(f"Could not read {path}")
        if arr.ndim == 2: arr = arr[..., None]
        elif arr.shape[-1] >= 3: arr = cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)
    arr = np.asarray(arr)
    if arr.ndim == 2: arr = arr[..., None]
    if BAND_INDICES is not None: arr = arr[..., BAND_INDICES]
    return arr.astype(np.float32)

def read_mask(path):
    suffix = Path(path).suffix.lower()
    if suffix == ".npy":
        m = np.load(path)
    elif suffix in {".tif", ".tiff"}:
        import rasterio
        with rasterio.open(path) as src: m = src.read(1)
    else:
        import cv2
        m = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if m is None: raise ValueError(f"Could not read {path}")
        if m.ndim == 3:
            rgb = cv2.cvtColor(m, cv2.COLOR_BGR2RGB)
            if RGB_MASK_MAP is None:
                raise ValueError("RGB mask found. Define RGB_MASK_MAP before loading.")
            out = np.full(rgb.shape[:2], IGNORE_INDEX, dtype=np.uint8)
            for color, cls in RGB_MASK_MAP.items():
                out[np.all(rgb == np.array(color), axis=-1)] = cls
            m = out
    if m.ndim == 3: m = m[..., 0]
    return np.asarray(m).astype(np.int64)

def radiometric_correction(x, low=1, high=99):
    # Per-band robust percentile stretch; preserves relative spatial structure.
    y = np.empty_like(x, dtype=np.float32)
    for c in range(x.shape[-1]):
        band = x[..., c]
        lo, hi = np.nanpercentile(band, [low, high])
        y[..., c] = np.clip((band - lo) / (hi - lo + 1e-6), 0, 1)
    return np.nan_to_num(y, nan=0.0, posinf=1.0, neginf=0.0)

def denoise_image(x):
    if not APPLY_DENOISING: return x
    import cv2
    # Apply NLM to each band independently; robust for multispectral inputs.
    out = np.empty_like(x)
    for c in range(x.shape[-1]):
        b = x[..., c].astype(np.float32)
        b8 = np.uint8(np.clip(b, 0, 1) * 255)
        out[..., c] = cv2.fastNlMeansDenoising(b8, None, 5, 7, 21).astype(np.float32) / 255.
    return out

def texture_and_morphology(x):
    gray = np.mean(x[..., :min(3, x.shape[-1])], axis=-1)
    q = np.uint8(np.clip(gray, 0, 1) * (GLCM_LEVELS - 1))
    glcm = graycomatrix(q, distances=[1, 2], angles=[0, np.pi/4, np.pi/2],
                        levels=GLCM_LEVELS, symmetric=True, normed=True)
    feats = []
    for prop in ["contrast", "homogeneity", "energy", "correlation"]:
        v = graycoprops(glcm, prop)
        feats.append(np.full(gray.shape, float(np.mean(v)), dtype=np.float32))
    morph = []
    for r in MORPH_RADII:
        se = disk(r)
        morph.extend([opening(gray, se), closing(gray, se),
                      white_tophat(gray, se), black_tophat(gray, se)])
    extra = np.stack(feats + morph, axis=-1).astype(np.float32)
    return extra

def resize_pair(image, mask, size):
    import cv2
    h, w = size, size
    image = cv2.resize(image, (w, h), interpolation=cv2.INTER_LINEAR)
    if image.ndim == 2: image = image[..., None]
    mask = cv2.resize(mask.astype(np.int32), (w, h), interpolation=cv2.INTER_NEAREST)
    return image, mask

## 6. Train/validation/test split

In [ ]:
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(pairs))
n_test = max(1, int(len(indices) * TEST_FRACTION))
n_val = max(1, int(len(indices) * VAL_FRACTION))
test_ids = indices[:n_test]
val_ids = indices[n_test:n_test+n_val]
train_ids = indices[n_test+n_val:]
assert len(train_ids) > 0, "Not enough samples for the selected split fractions."

train_pairs = [pairs[i] for i in train_ids]
val_pairs = [pairs[i] for i in val_ids]
test_pairs = [pairs[i] for i in test_ids]
print("Split:", len(train_pairs), len(val_pairs), len(test_pairs))

## 7. PyTorch dataset and augmentation

In [ ]:
class RemoteSensingDataset(Dataset):
    def __init__(self, samples, train=False):
        self.samples, self.train = samples, train
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        ip, mp = self.samples[idx]
        x = read_image(ip)
        y = read_mask(mp)
        x = radiometric_correction(x)
        x = denoise_image(x)
        x, y = resize_pair(x, y, PATCH_SIZE)
        aux = texture_and_morphology(x)
        x = np.concatenate([x, aux], axis=-1)
        if self.train:
            if random.random() < 0.5:
                x, y = np.flip(x, 1).copy(), np.flip(y, 1).copy()
            if random.random() < 0.5:
                x, y = np.flip(x, 0).copy(), np.flip(y, 0).copy()
            if random.random() < 0.5:
                x = np.rot90(x).copy(); y = np.rot90(y).copy()
        x = np.nan_to_num(x, nan=0.0, posinf=1.0, neginf=0.0)
        return torch.from_numpy(x.transpose(2,0,1).astype(np.float32)), torch.from_numpy(y.astype(np.int64))

train_ds = RemoteSensingDataset(train_pairs, train=True)
val_ds = RemoteSensingDataset(val_pairs, train=False)
test_ds = RemoteSensingDataset(test_pairs, train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

sample_x, sample_y = train_ds[0]
IN_CHANNELS = sample_x.shape[0]
print("Input channels:", IN_CHANNELS, "| Image:", tuple(sample_x.shape), "| Mask:", tuple(sample_y.shape))
print("Observed mask labels:", np.unique(sample_y.numpy())[:30])

## 8. Proposed model components: CNN + selective state-space-inspired encoder

In [ ]:
class ConvBNAct(nn.Module):
    def __init__(self, cin, cout, k=3, stride=1):
        super().__init__()
        self.block = nn.Sequential(nn.Conv2d(cin, cout, k, stride, k//2, bias=False),
                                   nn.BatchNorm2d(cout), nn.GELU())
    def forward(self, x): return self.block(x)

class ResidualConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.c1 = ConvBNAct(cin, cout)
        self.c2 = nn.Sequential(nn.Conv2d(cout, cout, 3, padding=1, bias=False),
                                nn.BatchNorm2d(cout))
        self.skip = nn.Identity() if cin == cout else nn.Conv2d(cin, cout, 1, bias=False)
    def forward(self, x): return F.gelu(self.c2(self.c1(x)) + self.skip(x))

class SelectiveSSM2D(nn.Module):
    # Efficient gated long-range sequence mixing over flattened spatial tokens.
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(1, channels)
        self.in_proj = nn.Conv2d(channels, channels*2, 1)
        self.dw = nn.Conv1d(channels, channels, 5, padding=2, groups=channels)
        self.select = nn.Conv1d(channels, channels, 1)
        self.out = nn.Conv2d(channels, channels, 1)
    def forward(self, x):
        z = self.norm(x)
        a, g = self.in_proj(z).chunk(2, dim=1)
        b, c, h, w = a.shape
        seq = a.flatten(2)
        mixed = self.dw(seq)
        gate = torch.sigmoid(self.select(seq))
        y = (mixed * gate).reshape(b, c, h, w)
        return x + self.out(y * torch.sigmoid(g))

class MultiScaleEncoder(nn.Module):
    def __init__(self, in_ch, base=32):
        super().__init__()
        self.e1 = ResidualConv(in_ch, base)
        self.e2 = ResidualConv(base, base*2)
        self.e3 = ResidualConv(base*2, base*4)
        self.e4 = ResidualConv(base*4, base*8)
        self.ssm = SelectiveSSM2D(base*8)
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        f1 = self.e1(x)
        f2 = self.e2(self.pool(f1))
        f3 = self.e3(self.pool(f2))
        f4 = self.ssm(self.e4(self.pool(f3)))
        return f1, f2, f3, f4

class DecoderBlock(nn.Module):
    def __init__(self, cin, skip, cout):
        super().__init__()
        self.conv = ResidualConv(cin+skip, cout)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

## 9. Novelty 1: multi-order fractional spectral–spatial interaction (FFSSFI)

In [ ]:
class FractionalSpectralSpatialInteraction(nn.Module):
    def __init__(self, channels, orders=(0.25, 0.50, 0.75, 1.00)):
        super().__init__()
        self.orders = tuple(float(v) for v in orders)
        self.order_logits = nn.Parameter(torch.zeros(len(self.orders)))
        self.projections = nn.ModuleList([nn.Conv2d(channels, channels, 1) for _ in self.orders])
        self.gate = nn.Sequential(nn.Conv2d(channels*2, channels, 1), nn.Sigmoid())
        self.fuse = nn.Sequential(nn.Conv2d(channels*2, channels, 1), nn.BatchNorm2d(channels), nn.GELU())
    def fractional_spectrum(self, x, order):
        # Differentiable fractional spectral phase operator, applied in 2-D Fourier space.
        # order=1 gives ordinary Fourier-domain representation; intermediate orders
        # continuously modulate phase to expose complementary frequency patterns.
        X = torch.fft.fft2(x.float(), norm="ortho")
        h, w = x.shape[-2:]
        fy = torch.fft.fftfreq(h, device=x.device).view(1,1,h,1)
        fx = torch.fft.fftfreq(w, device=x.device).view(1,1,1,w)
        radius = fx.square() + fy.square()
        phase = torch.exp((-1j * math.pi * float(order)) * radius)
        return torch.fft.ifft2(X * phase, norm="ortho").real
    def forward(self, x):
        weights = torch.softmax(self.order_logits, dim=0)
        responses = [proj(self.fractional_spectrum(x, order)) for proj, order in zip(self.projections, self.orders)]
        spectral = sum(w*r for w, r in zip(weights, responses))
        gate = self.gate(torch.cat([x, spectral], dim=1))
        interacted = gate * x + (1-gate) * spectral
        return self.fuse(torch.cat([x, interacted], dim=1)), weights

## 10. Novelty 2: Wasserstein-evidential uncertainty-guided optimization (WEUSO)

In [ ]:
class FFSSFI_WEUSO_Net(nn.Module):
    def __init__(self, in_ch, num_classes, base=32):
        super().__init__()
        self.encoder = MultiScaleEncoder(in_ch, base)
        self.ffssfi = FractionalSpectralSpatialInteraction(base*8)
        self.d3 = DecoderBlock(base*8, base*4, base*4)
        self.d2 = DecoderBlock(base*4, base*2, base*2)
        self.d1 = DecoderBlock(base*2, base, base)
        self.logit_head = nn.Conv2d(base, num_classes, 1)
        self.evidence_head = nn.Conv2d(base, num_classes, 1)
        self.feature_head = nn.Conv2d(base*8, 64, 1)
    def forward(self, x):
        f1, f2, f3, f4 = self.encoder(x)
        f4, order_weights = self.ffssfi(f4)
        deep_features = self.feature_head(f4)
        d = self.d3(f4, f3)
        d = self.d2(d, f2)
        d = self.d1(d, f1)
        logits = self.logit_head(d)
        evidence = F.softplus(self.evidence_head(d)) + 1e-6
        return {"logits": logits, "evidence": evidence,
                "alpha": evidence + 1.0, "deep_features": deep_features,
                "order_weights": order_weights}

model = FFSSFI_WEUSO_Net(IN_CHANNELS, NUM_CLASSES).to(DEVICE)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 11. Losses: segmentation + evidential uncertainty + Wasserstein alignment

In [ ]:
def dice_loss_from_logits(logits, target, num_classes, ignore_index=255):
    probs = torch.softmax(logits, dim=1)
    valid = target != ignore_index
    safe = target.clamp(0, num_classes-1)
    onehot = F.one_hot(safe, num_classes).permute(0,3,1,2).float()
    valid = valid.unsqueeze(1)
    probs, onehot = probs*valid, onehot*valid
    dims = (0,2,3)
    inter = (probs*onehot).sum(dims)
    denom = probs.sum(dims) + onehot.sum(dims)
    present = onehot.sum(dims) > 0
    dice = (2*inter + 1e-6)/(denom + 1e-6)
    return 1 - dice[present].mean() if present.any() else logits.sum()*0

def evidential_loss(alpha, target, num_classes, epoch, anneal_epochs=20, ignore_index=255):
    valid = target != ignore_index
    safe = target.clamp(0, num_classes-1)
    y = F.one_hot(safe, num_classes).permute(0,3,1,2).float()
    S = alpha.sum(dim=1, keepdim=True)
    probs = alpha / S
    err = (y-probs).square()
    variance = alpha*(S-alpha)/(S.square()*(S+1)+1e-8)
    data_fit = (err + variance).sum(dim=1)
    # Penalize non-target evidence to discourage unjustified confidence.
    adjusted = y + (1-y)*alpha
    kl = torch.lgamma(adjusted.sum(dim=1)) - torch.lgamma(adjusted).sum(dim=1)
    kl = kl + ((adjusted-1)*(torch.digamma(adjusted)-torch.digamma(adjusted.sum(dim=1, keepdim=True)))).sum(dim=1)
    anneal = min(1.0, float(epoch)/max(1, anneal_epochs))
    loss = data_fit + anneal*kl
    return loss[valid].mean() if valid.any() else alpha.sum()*0

def wasserstein_prototype_loss(features, target, num_classes, ignore_index=255):
    # Empirical 1-D Wasserstein-1 approximation: mean absolute difference
    # between normalized feature distributions and class prototype distributions.
    b, c, h, w = features.shape
    feat = F.interpolate(features, size=target.shape[-2:], mode="bilinear", align_corners=False)
    feat = F.normalize(feat, dim=1)
    losses = []
    for cls in range(num_classes):
        mask = (target == cls) & (target != ignore_index)
        if mask.any():
            vals = feat.permute(0,2,3,1)[mask]
            proto = F.normalize(vals.mean(0, keepdim=True), dim=1)
            losses.append(torch.abs(vals-proto).mean())
    return torch.stack(losses).mean() if losses else features.sum()*0

def total_loss(outputs, target, epoch):
    logits, alpha = outputs["logits"], outputs["alpha"]
    ce = F.cross_entropy(logits, target, ignore_index=IGNORE_INDEX)
    dice = dice_loss_from_logits(logits, target, NUM_CLASSES, IGNORE_INDEX)
    evid = evidential_loss(alpha, target, NUM_CLASSES, epoch, ignore_index=IGNORE_INDEX)
    wass = wasserstein_prototype_loss(outputs["deep_features"], target, NUM_CLASSES, IGNORE_INDEX)
    loss = ce + dice + 0.20*evid + 0.05*wass
    return loss, {"ce": ce.detach(), "dice": dice.detach(), "evidential": evid.detach(), "wasserstein": wass.detach()}

## 12. Uncertainty-guided pixel refinement

In [ ]:
@torch.no_grad()
def uncertainty_guided_refinement(outputs, strength=0.35):
    alpha = outputs["alpha"]
    probs = alpha / alpha.sum(dim=1, keepdim=True)
    uncertainty = NUM_CLASSES / alpha.sum(dim=1, keepdim=True)
    # Local consensus is used only to refine high-uncertainty pixels.
    local = F.avg_pool2d(probs, kernel_size=3, stride=1, padding=1)
    refined = (1-strength*uncertainty)*probs + strength*uncertainty*local
    refined = refined / refined.sum(dim=1, keepdim=True).clamp_min(1e-8)
    return refined, uncertainty

## 13. Training and validation

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

@torch.no_grad()
def evaluate_loss(net, loader, epoch=0):
    net.eval()
    vals=[]
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        out=net(x)
        loss,_=total_loss(out,y,epoch)
        vals.append(loss.item())
    return float(np.mean(vals)) if vals else np.nan

history=[]
best_val=float("inf")
for epoch in range(1, EPOCHS+1):
    model.train()
    running=[]
    pbar=tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
    for x,y in pbar:
        x,y=x.to(DEVICE, non_blocking=True),y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type=="cuda")):
            out=model(x)
            loss,parts=total_loss(out,y,epoch)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        running.append(loss.item())
        pbar.set_postfix(loss=f"{np.mean(running):.4f}")
    val_loss=evaluate_loss(model,val_loader,epoch)
    scheduler.step()
    train_loss=float(np.mean(running))
    history.append({"epoch":epoch,"train_loss":train_loss,"val_loss":val_loss,
                    "lr":optimizer.param_groups[0]["lr"]})
    print(f"Epoch {epoch:03d} | train {train_loss:.4f} | val {val_loss:.4f}")
    if val_loss < best_val:
        best_val=val_loss
        torch.save({"model":model.state_dict(),"epoch":epoch,"val_loss":val_loss,
                    "class_names":CLASS_NAMES,"in_channels":IN_CHANNELS}, "best_ffssfi_weuso.pt")

history_df=pd.DataFrame(history)
display(history_df.tail())

## 14. Learning curves

In [ ]:
fig, ax = plt.subplots(figsize=(8,5), dpi=150)
ax.plot(history_df.epoch, history_df.train_loss, label="Training")
ax.plot(history_df.epoch, history_df.val_loss, label="Validation")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("FFSSFI–WEUSO training convergence")
ax.grid(alpha=.25); ax.legend(); plt.tight_layout(); plt.show()

## 15. Test metrics: OA, mean precision/recall/F1, mIoU, per-class IoU, uncertainty

In [ ]:
ckpt=torch.load("best_ffssfi_weuso.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()

all_true, all_pred, all_unc = [], [], []
with torch.no_grad():
    for x,y in tqdm(test_loader, desc="Testing"):
        x=x.to(DEVICE)
        out=model(x)
        probs, unc=uncertainty_guided_refinement(out)
        pred=probs.argmax(1).cpu().numpy()
        all_pred.append(pred.reshape(-1))
        all_true.append(y.numpy().reshape(-1))
        all_unc.append(unc.cpu().numpy().reshape(-1))

yt=np.concatenate(all_true); yp=np.concatenate(all_pred); uu=np.concatenate(all_unc)
valid=yt != IGNORE_INDEX
yt,yp=yt[valid],yp[valid]
labels=list(range(NUM_CLASSES))
cm=confusion_matrix(yt,yp,labels=labels)
precision,recall,f1,support=precision_recall_fscore_support(yt,yp,labels=labels,zero_division=0)
iou=np.diag(cm)/(cm.sum(1)+cm.sum(0)-np.diag(cm)+1e-9)
metrics={
    "Overall Accuracy":accuracy_score(yt,yp),
    "Mean Precision":float(np.mean(precision)),
    "Mean Recall":float(np.mean(recall)),
    "Mean F1":float(np.mean(f1)),
    "Mean IoU":float(np.mean(iou)),
    "Pixel Error Rate":1-accuracy_score(yt,yp),
    "Mean Predictive Uncertainty":float(np.mean(uu[uu < 1.01])) if len(uu) else np.nan
}
display(pd.DataFrame({"Class":CLASS_NAMES,"Precision":precision,"Recall":recall,"F1":f1,"IoU":iou,"Support":support}))
display(pd.DataFrame([metrics]).T.rename(columns={0:"Value"}))

## 16. Confusion matrix

In [ ]:
fig,ax=plt.subplots(figsize=(8,7),dpi=160)
im=ax.imshow(cm,cmap="Blues")
ax.set_xticks(range(NUM_CLASSES),CLASS_NAMES,rotation=45,ha="right")
ax.set_yticks(range(NUM_CLASSES),CLASS_NAMES)
ax.set_xlabel("Predicted class"); ax.set_ylabel("Ground truth"); ax.set_title("Pixel-level confusion matrix")
fig.colorbar(im,ax=ax,fraction=.046,pad=.04)
threshold=cm.max()/2 if cm.size else 0
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j,i,f"{cm[i,j]:,}",ha="center",va="center",
                color="white" if cm[i,j]>threshold else "black",fontsize=8)
plt.tight_layout(); plt.show()

## 17. Segmentation visualizations

In [ ]:
def show_prediction(index=0):
    x,y=test_ds[index]
    model.eval()
    with torch.no_grad():
        out=model(x.unsqueeze(0).to(DEVICE))
        probs,unc=uncertainty_guided_refinement(out)
        pred=probs.argmax(1)[0].cpu().numpy()
        unc=unc[0,0].cpu().numpy()
    # Display first three channels as RGB where possible; otherwise first band in grayscale.
    raw=read_image(test_pairs[index][0])
    raw=radiometric_correction(raw)
    if raw.shape[-1]>=3:
        rgb=raw[...,:3]
        rgb=np.clip(rgb,0,1)
    else:
        rgb=np.repeat(raw[...,:1],3,axis=-1)
    import cv2
    rgb=cv2.resize(rgb,(PATCH_SIZE,PATCH_SIZE))
    fig,axs=plt.subplots(1,4,figsize=(17,4),dpi=150)
    axs[0].imshow(rgb); axs[0].set_title("Input image")
    axs[1].imshow(y,cmap="nipy_spectral",vmin=0,vmax=NUM_CLASSES-1); axs[1].set_title("Ground truth")
    axs[2].imshow(pred,cmap="nipy_spectral",vmin=0,vmax=NUM_CLASSES-1); axs[2].set_title("FFSSFI–WEUSO")
    axs[3].imshow(unc,cmap="magma",vmin=0,vmax=1); axs[3].set_title("Predictive uncertainty")
    for a in axs: a.axis("off")
    plt.tight_layout(); plt.show()

show_prediction(0)

## 18. Reference U-Net baseline and common evaluation interface

In [ ]:
class UNetBaseline(nn.Module):
    def __init__(self, in_ch, num_classes, base=32):
        super().__init__()
        self.enc=MultiScaleEncoder(in_ch,base)
        self.d3=DecoderBlock(base*8,base*4,base*4)
        self.d2=DecoderBlock(base*4,base*2,base*2)
        self.d1=DecoderBlock(base*2,base,base)
        self.head=nn.Conv2d(base,num_classes,1)
    def forward(self,x):
        a,b,c,d=self.enc(x)
        z=self.d3(d,c); z=self.d2(z,b); z=self.d1(z,a)
        return {"logits":self.head(z)}

def train_reference_baseline(net, epochs=30):
    net=net.to(DEVICE)
    opt=torch.optim.AdamW(net.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    for ep in range(1,epochs+1):
        net.train(); losses=[]
        for x,y in train_loader:
            x,y=x.to(DEVICE),y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits=net(x)["logits"]
            loss=F.cross_entropy(logits,y,ignore_index=IGNORE_INDEX)+dice_loss_from_logits(logits,y,NUM_CLASSES)
            loss.backward(); opt.step(); losses.append(loss.item())
        print(f"Baseline epoch {ep}/{epochs}: loss={np.mean(losses):.4f}")
    return net

def evaluate_logits_model(net):
    net.eval(); t,p=[],[]
    with torch.no_grad():
        for x,y in test_loader:
            logits=net(x.to(DEVICE))["logits"]
            t.append(y.numpy().reshape(-1)); p.append(logits.argmax(1).cpu().numpy().reshape(-1))
    t=np.concatenate(t); p=np.concatenate(p); m=t!=IGNORE_INDEX; t,p=t[m],p[m]
    iou=jaccard_score(t,p,labels=list(range(NUM_CLASSES)),average=None,zero_division=0)
    pr,re,f1,_=precision_recall_fscore_support(t,p,labels=list(range(NUM_CLASSES)),average="macro",zero_division=0)
    return {"OA":accuracy_score(t,p),"Macro Precision":pr,"Macro Recall":re,"Macro F1":f1,"mIoU":float(np.mean(iou))}

# Example:
# unet = train_reference_baseline(UNetBaseline(IN_CHANNELS, NUM_CLASSES), epochs=30)
# print(evaluate_logits_model(unet))

## 19. Ablation switches for FFSSFI and WEUSO

In [ ]:
# For controlled ablations, train separate runs with these settings.
# Full model: FFSSFI enabled, evidential coefficient=0.20, Wasserstein coefficient=0.05,
# uncertainty refinement strength=0.35.
#
# Suggested ablations:
# A1: remove FFSSFI (replace f4, _ = self.ffssfi(f4) with f4 unchanged)
# A2: remove Wasserstein loss (set its coefficient to 0)
# A3: remove evidential loss (set its coefficient to 0)
# A4: remove uncertainty-guided refinement (use softmax(logits))
# A5: single FrFT order (orders=(1.0,))
#
# Keep the same split, epochs, optimizer, augmentations and random seed for fair comparison.

## 20. Export metric tables

In [ ]:
# Export only when required; paths are relative to the current notebook directory.
metrics_df=pd.DataFrame([metrics])
class_metrics_df=pd.DataFrame({"Class":CLASS_NAMES,"Precision":precision,"Recall":recall,"F1":f1,"IoU":iou,"Support":support})
# metrics_df.to_csv("ffssfi_weuso_overall_metrics.csv",index=False)
# class_metrics_df.to_csv("ffssfi_weuso_class_metrics.csv",index=False)
# history_df.to_csv("ffssfi_weuso_training_history.csv",index=False)

## Execution checklist
1. Upload/extract the dataset and set `DATA_ROOT`, `CLASS_NAMES`, `BAND_INDICES` and (if needed) `RGB_MASK_MAP`.
2. Confirm the printed matched-pair count and observed mask labels.
3. Ensure mask IDs are contiguous integers `0 ... NUM_CLASSES-1` (or `IGNORE_INDEX` for unlabeled pixels). Remap IDs if necessary.
4. Run preprocessing, split, model, training and test cells in order.
5. Report test metrics from the held-out test set only. Do not manually fill performance values.
6. For faithful SCSM, PSseg-FD, CVMH-UNet and PENet comparisons, implement each architecture from its source paper and train each using the identical split and protocol.